## ATL06 Tiled Product Demonstration

In [ ]:
from sliderule import sliderule, s2tile
import geopandas as gpd
import matplotlib.pyplot as plt
import time

### Define Areas of Interest

In [ ]:
greenland_poly_50km2 = [
    {"lon": -27.25979597144949,     "lat": 73.49716901328901},
    {"lon": -27.25979597144949,     "lat": 73.56076480273342},
    {"lon": -27.48407173814204,     "lat": 73.56076480273342},
    {"lon": -27.48407173814204,     "lat": 73.49716901328901},
    {"lon": -27.25979597144949,     "lat": 73.49716901328901}
]
greenland_poly_300km2 = [
    {"lon": -27.219435431853217,    "lat": 73.49818164811984},
    {"lon": -27.219435431853217,    "lat": 73.64424276904958},
    {"lon": -27.80812065118551,     "lat": 73.64424276904958},
    {"lon": -27.80812065118551,     "lat": 73.49818164811984},
    {"lon": -27.219435431853217,    "lat": 73.49818164811984}
]
greenland_convex_poly_450km2 = [
    {"lon": -29.127476024254214,    "lat": 73.73869799255206},
    {"lon": -28.57131232009959,     "lat": 73.68944445330746},
    {"lon": -28.398006201999944,    "lat": 73.76409914046496},
    {"lon": -28.436079389819962,    "lat": 73.850847588515},
    {"lon": -28.789962359255153,    "lat": 73.91373599019127},
    {"lon": -29.256954649831314,    "lat": 73.86000697128824},
    {"lon": -29.127476024254214,    "lat": 73.73869799255206}
]
greenland_poly_5000km2 = [
    {"lon": -27.293532869741416,    "lat": 73.48761834293555},
    {"lon": -27.293532869741416,    "lat": 74.0798487786565},
    {"lon": -29.741987353437068,    "lat": 74.0798487786565},
    {"lon": -29.741987353437068,    "lat": 73.48761834293555},
    {"lon": -27.293532869741416,    "lat": 73.48761834293555}
]
aois = [greenland_poly_50km2, greenland_poly_300km2, greenland_convex_poly_450km2, greenland_poly_5000km2]
names = ["Bbox50km2", "Bbox300km2", "Convex450km2", "Bbox5000km2"]
regions = {names[i]: sliderule.toregion(aois[i]) for i in range(len(aois))}

### Display Areas of Interest

In [ ]:
# NSIDC Sea Ice Polar Stereographic North
crs = "EPSG:3413"
countries = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
greenland = countries[countries["ADMIN"] == "Greenland"].to_crs(crs)

labels = names
colors = ["tab:red", "tab:orange", "tab:blue", "tab:green"]
region_gdfs = [r["gdf"].set_crs("EPSG:4326", allow_override=True).to_crs(crs) for r in regions.values()]

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(14, 8))

# Full Greenland view (regions are small, so mark them with an outline of the largest)
greenland.plot(ax=ax_full, color="whitesmoke", edgecolor="black", linewidth=0.5)
for gdf, color, label in zip(region_gdfs, colors, labels):
    gdf.boundary.plot(ax=ax_full, color=color, linewidth=1.5, label=label)
ax_full.set_title("Greenland")
ax_full.set_axis_off()
ax_full.legend(loc="lower right")

# Zoomed view around the regions
greenland.plot(ax=ax_zoom, color="whitesmoke", edgecolor="black", linewidth=0.5)
for gdf, color, label in zip(region_gdfs, colors, labels):
    gdf.plot(ax=ax_zoom, facecolor=color, edgecolor=color, alpha=0.3)
    gdf.boundary.plot(ax=ax_zoom, color=color, linewidth=1.5, label=label)
xmin, ymin, xmax, ymax = region_gdfs[len(labels) - 1].total_bounds
pad = 0.25 * max(xmax - xmin, ymax - ymin)
ax_zoom.set_xlim(xmin - pad, xmax + pad)
ax_zoom.set_ylim(ymin - pad, ymax + pad)
ax_zoom.set_title("Regions (zoomed)")
ax_zoom.set_axis_off()
ax_zoom.legend(loc="lower right")

# Box on the full map showing the zoom extent
ax_full.add_patch(plt.Rectangle((xmin - pad, ymin - pad), xmax - xmin + 2 * pad, ymax - ymin + 2 * pad,
                                fill=False, edgecolor="black", linestyle="--"))

plt.tight_layout()
plt.show()

### Read Data

In [ ]:
BUCKET = "sliderule-public"
PREFIX = "atl06t"

In [ ]:
def S2TileSubset(reader, name, aoi, start_time=None, end_time=None):
    start = time.time()
    gdf = reader.read_tiles(aoi, start_time, end_time)
    print(f"S2TileReader subsetted {name} / {len(gdf)} rows in {time.time() - start:.2f} seconds")

In [ ]:
reader = s2tile.Reader(BUCKET, PREFIX)

# S2Tile - Spatial
for i in range(len(names)):
    S2TileSubset(reader, names[i], aois[i])

# S2Tile - Temporal
S2TileSubset(reader, names[1], aois[1], "2020-08-27", "2020-08-28")
S2TileSubset(reader, names[1], aois[1], "2020-08-27", "2020-11-28")
S2TileSubset(reader, names[1], aois[1], "2020-08-27", "2021-08-28")


In [ ]:
gdf